In [ ]:
import ee
import geemap

try:
    ee.Initialize()
    print("Google Earth Engine initialized successfully!")
except Exception as e:
    print("Authentication required. Launching browser link...")
    ee.Authenticate()

Authentication required. Launching browser link...



Successfully saved authorization token.


EEException: Please authorize access to your Earth Engine account by running

earthengine authenticate

in your command line, or ee.Authenticate() in Python, and then retry.

In [3]:
4/1AdkVLPyITn-pDTZhbT54QkTa5-k815EloYULrvnCoeSw54PKSNqgzpd9Rdc4/1AdkVLPwb6dVqlOegnVH6RnniR6JMyeNSA5X_8i9L2S0au2hnO6DFctn_LMk


SyntaxError: invalid decimal literal (2801985315.py, line 1)

In [6]:
4/1AdkVLPyITn-pDTZhbT54QkTa5-k815EloYULrvnCoeSw54PKSNqgzpd9Rdc4/1AdkVLPwb6dVqlOegnVH6RnniR6JMyeNSA5X_8i9L2S0au2hnO6DFctn_LMk


SyntaxError: invalid decimal literal (2801985315.py, line 1)

In [5]:
import ee
import geemap

project_id = 'climate-resilition'

try:
    # Initialize using your newly activated project container
    ee.Initialize(project=project_id)
    print(f"🎉 SUCCESS! Connected to Earth Engine via project: {project_id}")
    
    # Render the interactive map framework layout
    Map = geemap.Map(center=[22.0, 78.0], zoom=5)
    display(Map)

except Exception as e:
    print("\nInitialization check running... Error details:")
    print(e)

🎉 SUCCESS! Connected to Earth Engine via project: climate-resilition


Map(center=[22.0, 78.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [7]:
import ee
import geemap

# 1. Initialize Map view and define project id
project_id = 'climate-resilition'
ee.Initialize(project=project_id)

# 2. Target Area (ROI): Bounding box over an area in India (e.g., Central/Western states)
roi = ee.Geometry.Rectangle([72.0, 15.0, 82.0, 25.0]) 

# 3. SLOW MOVING CRITERIA: Terrain Slope (Avoid steep terrain for infrastructure stabilization)
dem = ee.Image('USGS/SRTMGL1_003').clip(roi)
slope = ee.Terrain.slope(dem)
slope_suitability = slope.expression('1 - (b(0) / 20)', {'b': slope}).clamp(0, 1)

# 4. FAST MOVING CRITERIA: Recent Heavy Rain Accumulation Stress (Climate hazard monitoring)
monsoon_rain = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY') \
    .filterBounds(roi) \
    .filterDate('2025-06-01', '2025-09-30') \
    .sum()
rain_suitability = monsoon_rain.expression('1 - (b(0) / 2000)', {'b': monsoon_rain}).clamp(0, 1)

# 5. WEIGHTED COMBINATION (AHP Analytical Weights Matrix)
# 40% Weight to Slope Safety, 60% Weight to Climate Rain Hazard Prevention
final_suitability = slope_suitability.multiply(0.40).add(rain_suitability.multiply(0.60))

# 6. Render Layers on Map Frame
Map = geemap.Map(center=[20.0, 77.0], zoom=5)
suitability_vis = {
    'min': 0, 
    'max': 1, 
    'palette': ['#d73027', '#fee08b', '#1a9850'] # Red (High Hazard) to Green (Safe Settlement Zone)
}
Map.addLayer(final_suitability, suitability_vis, 'Resilient Settlement Suitability')
display(Map)


Map(center=[20.0, 77.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [9]:
import ee
import geemap
from datetime import datetime, timedelta

# Ensure your authorized project id is referenced
project_id = 'climate-resilition'
ee.Initialize(project=project_id)

# 1. DEFINE DELHI BOUNDARIES (Region of Interest)
# Centered directly over Delhi National Capital Territory (NCT)
delhi_lon, delhi_lat = 77.2167, 28.6667
roi = ee.Geometry.Point([delhi_lon, delhi_lat]).buffer(30000) # 30km radius encompassing Delhi-NCR

# 2. CONFIGURING THE FAST-MOVING TIME WINDOW
# We pull data leading up to the current active July monsoon system
end_date = datetime.now()
start_date = end_date - timedelta(days=7) # 7-day rolling historical storm tracking window

start_str = start_date.strftime('%Y-%m-%d')
end_str = end_date.strftime('%Y-%m-%d')
print(f"Tracking fast-moving precipitation anomalies between {start_str} and {end_str}...")

# 3. STREAMING FAST-MOVING WEATHER DATA (CHIRPS Daily)
rain_collection = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY') \
    .filterBounds(roi) \
    .filterDate(start_str, end_str)

# Sum up total millimeters of rain dropped during this active storm window
total_rain = rain_collection.sum().clip(roi)

# 4. FAST-MOVING CLIMATE RESILIENCE SCORING
# Normalize: If a pixel receives >100mm in 7 days, it's a severe water-logging risk (Score 0)
# Low rainfall accumulation in low-lying built areas keeps the settlement score safe (Score 1)
rain_score = total_rain.expression(
    '1 - (b(0) / 100)', {'b': total_rain}
).clamp(0, 1)

# 5. RENDER THE INTERACTIVE DISASTER MONITORING ENVIRONMENT
Map = geemap.Map(center=[delhi_lat, delhi_lon], zoom=10)

# High-impact rainfall styling: Transparent/Blue to Dark Violet for extreme downpours
rain_vis = {
    'min': 0,
    'max': 100,
    'palette': ['#ffffff', '#bcbddc', '#74a9cf', '#3690c0', '#0570b0', '#023858', '#4d004b']
}

# Climate Suitability Index styling: Red (Severe Risk Zone) -> Yellow -> Green (Safe Settlement)
suitability_vis = {
    'min': 0,
    'max': 1,
    'palette': ['#d73027', '#f46d43', '#fdae61', '#fee08b', '#d9ef8b', '#a6d96a', '#1a9850']
}

# Display layers to the map view panel
Map.addLayer(total_rain, rain_vis, f'Total Rain (mm) Past 7 Days')
Map.addLayer(rain_score, suitability_vis, 'Dynamic Rain Suitability Index')

# Add a marker right over central Delhi
Map.add_marker([delhi_lat, delhi_lon], tooltip="Delhi Central Center Node")
display(Map)


Tracking fast-moving precipitation anomalies between 2026-07-01 and 2026-07-08...


EEException: Image.select: Invalid band number (0) specified to select. Input only contains 0 bands.

In [5]:
import ee
import geemap
from datetime import datetime, timedelta

project_id = 'climate-resilition'
ee.Initialize(project=project_id)

delhi_lon, delhi_lat = 77.2167, 28.6667
roi = ee.Geometry.Point([delhi_lon, delhi_lat]).buffer(60000)

end_date = datetime.now() - timedelta(days=2)
start_date = end_date - timedelta(days=5)

start_str = start_date.strftime('%Y-%m-%dT%H:%M:%S')
end_str = end_date.strftime('%Y-%m-%dT%H:%M:%S')

# 1. FETCH FAST-MOVING RAIN
rain_collection = ee.ImageCollection('NASA/GPM_L3/IMERG_V07') \
    .filterBounds(roi) \
    .filterDate(start_str, end_str) \
    .select('precipitation')
total_rain = rain_collection.reduce(ee.Reducer.sum()).multiply(0.5).clip(roi)

rain_score = total_rain.expression(
    '1 - (b("precipitation_sum") / 150)', {'b': total_rain}
).clamp(0, 1)

# 2. FETCH SLOW-MOVING TOPOGRAPHY (For detailed background texture)
dem = ee.Image('USGS/SRTMGL1_003').clip(roi)
slope = ee.Terrain.slope(dem)
# Normalize slope (flat land is preferred)
slope_score = slope.expression('1 - (b(0) / 15)', {'b': slope}).clamp(0, 1)

# 3. BLEND THEM (Multi-Criteria Suitability)
# 70% weight to rainfall accumulation, 30% to structural terrain slopes
final_engine_suitability = rain_score.multiply(0.7).add(slope_score.multiply(0.3))

# 4. INITIALIZE RENDER CONTROLS
Map = geemap.Map(center=[delhi_lat, delhi_lon], zoom=10)

# FIX: Added 'opacity': 0.45 so text and roads slice right through the colors cleanly
suitability_vis = {
    'min': 0,
    'max': 1,
    'palette': ['#d73027', '#fdae61', '#fee08b', '#a6d96a', '#1a9850'],
    'opacity': 0.8
}

# Display your sophisticated Multi-Criteria output
Map.addLayer(final_engine_suitability, suitability_vis, 'Final Resilient Settlement Suitability')
Map.add_marker([delhi_lat, delhi_lon], tooltip="Delhi Core")
display(Map)

Map(center=[28.6667, 77.2167], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

In [ ]:
import ee
import geemap.foliumap as geemap  # Optimized mapping engine for web browsers
import streamlit as st
from datetime import datetime, timedelta

# 1. Page Configuration
st.set_page_config(layout="wide", page_title="Delhi Climate Resilient Suitability Engine")
st.title("🗺️ Multi-Criteria Climate Suitability Engine")
st.subheader("Delhi-NCR Settlement & Infrastructure Planning Dashboard")

# 2. Authenticate and Initialize GEE
# Web servers require initialization without active window popups
project_id = 'climate-resilition'
try:
    ee.Initialize(project=project_id)
except Exception as e:
    st.error(f"Earth Engine Initialization Failed: {e}")

# 3. Sidebar Interactive Controls
st.sidebar.header("🎛️ Analysis Matrices Configuration")
rain_weight = st.sidebar.slider("Fast-Moving Rain Data Weight", 0.0, 1.0, 0.70, 0.05)
slope_weight = round(1.0 - rain_weight, 2)
st.sidebar.text(f"Slow-Moving Slope Weight: {slope_weight}")

# 4. Processing Pipelines
delhi_lon, delhi_lat = 77.2167, 28.6667
roi = ee.Geometry.Point([delhi_lon, delhi_lat]).buffer(30000)

end_date = datetime.now() - timedelta(days=2)
start_date = end_date - timedelta(days=5)

# Fast-Moving Data Ingest
rain_collection = ee.ImageCollection('NASA/GPM_L3/IMERG_V07') \
    .filterBounds(roi) \
    .filterDate(start_date.strftime('%Y-%m-%dT%H:%M:%S'), end_date.strftime('%Y-%m-%dT%H:%M:%S')) \
    .select('precipitation')

total_rain = rain_collection.reduce(ee.Reducer.sum()).multiply(0.5).clip(roi)
rain_score = total_rain.expression('1 - (b("precipitation_sum") / 150)', {'b': total_rain}).clamp(0, 1)

# Slow-Moving Data Ingest
dem = ee.Image('USGS/SRTMGL1_003').clip(roi)
slope = ee.Terrain.slope(dem)
slope_score = slope.expression('1 - (b(0) / 15)', {'b': slope}).clamp(0, 1)

# Combined Decision Engine
final_suitability = rain_score.multiply(rain_weight).add(slope_score.multiply(slope_weight))

# 5. Rendering Web Framework Mapping Controls
Map = geemap.Map(center=[delhi_lat, delhi_lon], zoom=10)

suitability_vis = {
    'min': 0,
    'max': 1,
    'palette': ['#d73027', '#fdae61', '#fee08b', '#a6d96a', '#1a9850'],
    'opacity': 0.6}

Map.addLayer(final_suitability, suitability_vis, 'Engine Suitability Index')
Map.add_marker([delhi_lat, delhi_lon], popup="Delhi Core")

# Push the interactive engine canvas straight onto the browser layout 
Map.to_streamlit(height=650)

: 